##  Notebook encargado de generar la base de datos de elecciones 1970

In [ ]:
# Imports y función para normalizar nombres de comunas

import pandas as pd
import unicodedata, re, os

BASE = "/Users/Angelo/Dropbox/Tesis 2026 ME"

def normalize(s):
    s = str(s).upper().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^A-Z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# Homologación y agregación de resultados electorales de 1970
ALIAS = {
    "AYSEN": "AISEN",
    "CANETE": "CANETE",
    "CONCEPCION": "CONCEPCION",
    "TALCAHUANO": "TALCAHUANO",
    "MARCHIGUE": "MARCHIHUE",
    "LAGO RANCO": "LAGO RANGO", 
    "PORVENIR": "PORVENIR",
    "ZUNIGA": "ZUNIGA",
    "PITRUFQUEN": "PITRUFQUEN",
    "QUEMCHI": "QUEMCHI",
    "FUTRONO": "FUTRONO",
    "RAUCO": "RAUCO",
    "TEODORO SCHMIDT": "TEODORO SCHMIDT",
    "SAN JUAN DE LA COSTA": "SAN JUAN DE LA COSTA",
    "INDEPENDENCIA": "INDEPENDENCIA",
    "CHIGUAYANTE": "CONCEPCION",  # daughter of Concepcion
    "PADRE LAS CASAS": "TEMUCO",  # daughter of Temuco
    "MACUL": "NUNOA",  # daughter of Ñuñoa
    "PUDAHUEL": "BARRANCAS",  # old name
    "LONTUE": "MOLINA",  # sub-commune of Molina
    "ANTUCO": "LOS ANGELES",  # sub-commune
    "CABRERO": "YUMBEL",  # actually separate - keep
    "CISNES": "AISEN",
    "BALMACEDA": "AISEN",
    "ENTRE LAGOS": "OSORNO",  # sub-commune
    "QUILLECO": "QUILECO",  # spelling variant
    "PINTO": "PINTO",
    "SAN PEDRO DE ALCANTARA": "ROSARIO",  # old name
    "COCHAMO": "PUERTO MONTT",  # sub-commune
    "ACHAO": "ACHAO",
    "EL MANZANO": "SAN FERNANDO",  # sub-commune
    "LAGUNILLAS": "CASABLANCA",  # sub-commune
    "CUNCUMEN": "SAN ANTONIO",  # sub-commune
    "CERRO SOMBRERO": "PRIMAVERA",  # maps to
    "CARAHUE": "CARAHUE",
}
# panel de referencia: market_access_comunas1970_sensibilidad generado en notebook anteríor. 
panel = pd.read_stata(f"{BASE}/Procesos Tesis/Data/market_access/market_access_comunas1970_sensibilidad.dta")
panel["com_norm"] = panel["comuna"].apply(normalize)
panel_names = set(panel["com_norm"].unique())

#  base elecciones presidenciales  1970
pdf = pd.read_csv(f"{BASE}/Codes/elecciones_1970_comunal_pdf.csv")
pdf["match_name"] = pdf["comuna_norm"].map(ALIAS).fillna(pdf["comuna_norm"])

# ── Check alias  ──
for pdf_name, panel_name in ALIAS.items():
    if panel_name not in panel_names and panel_name != pdf_name:
        print(f"   Alias '{pdf_name}' → '{panel_name}' NOT FOUND in panel")

agg_rows = []
for name, grp in pdf.groupby("match_name"):
    total_votos = grp["total_votos"].sum()
    if total_votos > 0:
        pct_t = (grp["pct_tomic"] * grp["total_votos"]).sum() / total_votos
        pct_a = (grp["pct_alessandri"] * grp["total_votos"]).sum() / total_votos
        pct_al = (grp["pct_allende"] * grp["total_votos"]).sum() / total_votos
    else:
        pct_t = pct_a = pct_al = 0
    agg_rows.append({
        "com_norm": name,
        "pct_tomic_70": round(pct_t, 2),
        "pct_alessandri_70": round(pct_a, 2),
        "pct_allende_70": round(pct_al, 2),
        "total_votos_70": total_votos,
        "source": "pdf_ocr"
    })

pdf_agg = pd.DataFrame(agg_rows)

In [ ]:
# Integración de los resultados electorales con el panel comunal

merged = panel.merge(pdf_agg, on="com_norm", how="left")

n_matched = merged["source"].notna().sum()
n_missing = merged["source"].isna().sum()

print(f"\n=== MERGE RESULTS ===")
print(f"Panel communes: {len(panel)}")
print(f"PDF communes (after aggregation): {len(pdf_agg)}")
print(f"Matched to panel: {n_matched}")
print(f"Missing (no PDF data): {n_missing}")

# Verificación de missings
missing = merged[merged["source"].isna()]["com_norm"].tolist()
print(f"\nMissing communes ({n_missing}):")
for m in sorted(missing):
    print(f"  {m}")


# Import data final elecciones 1970.
out = f"{BASE}/Codes/panel_con_elecciones_1970_corregidas.csv"
# Only save key columns
keep = ["comuna", "com_norm", "pct_tomic_70", "pct_alessandri_70", 
        "pct_allende_70", "total_votos_70", "source"]
merged[keep].to_csv(out, index=False, encoding="utf-8")